# Agent高级用法

## 1. 设置Agent名称

In [2]:
import os
from langchain.agents import create_agent
from langchain_classic.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv()

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model = "gpt-5-mini",
    api_key = CLOSEAI_API_KEY,
    base_url = CLOSEAI_BASE_URL
)

# 使用内置工具
web_search = TavilySearch(
    max_results=2,
    tavily_api_key=os.getenv("TavilySearch_API_KEY")
)


agent = create_agent(
    model=model,
    tools=[web_search],
    name="Agent01"
)

response = agent.invoke({
    "messages": [
        # {"role": "system", "content": "你是一个查询天气的助手，根据用户提问查询天气。如果问题与天气无关，你可以说：“我不清楚这个问题的答案。”"},
        {"role": "user", "content": "武汉的天气如何？"}
    ]
})
# rprint(response)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

武汉的天气如何？
================================== Ai Message ==================================
Name: Agent01
Tool Calls:
  tavily_search (call_WP7Hott22M8NPJj3VSFWiixl)
 Call ID: call_WP7Hott22M8NPJj3VSFWiixl
  Args:
    query: 武汉 今天天气 现在 实时
    search_depth: fast
    include_images: False
    time_range: day
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "武汉 今天天气 现在 实时", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://m.tianqi.com/hebei-hanerzhuangxiang/32562/2026-10-01?qd=tq15", "title": "【迁西汉儿庄乡天气预报】汉儿庄乡天气预报一周_汉儿庄乡天气预报今天、7天、15天、30天、40天气预报查询—天气网", "content": "北京空气质量 天津空气质量 重庆空气质量 上海空气质量 石家庄空气质量 苏州空气质量 合肥空气质量 西安空气质量 厦门空气质量 无锡空气质量 兰州空气质量 哈尔滨空气质量 珠海空气质量 连云港空气质量 深圳空气质量 南京空气质量 成都空气质量 济宁空气质量 徐州空气质量 青岛空气质量 广州空气质量 杭州空气质量 郑州空气质量 武汉空气质量 内蒙古空气质量 台湾空气质量 澳门空气质量 香港空气质量 海南空气质量 广东空气质量 云南空气质量\n\n东京天气 胡志

## 2. 系统提示词设置

使用`system_prompt`参数设置，可以是字符串，也可以是`SystemMessage`

In [5]:
from langchain_core.tools import tool
import os
from rich import print as rprint
from langchain.agents import create_agent
from langchain_classic.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model = "gpt-5-mini",
    api_key = CLOSEAI_API_KEY,
    base_url = CLOSEAI_BASE_URL
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    查询天气工具

    Args:
        city: 具体的城市

    Returns:
        城市天气信息

    """
    return f"{city}天气晴朗，温度：15℃"

agent = create_agent(
    model=model,
    name="Agent01",
    tools=[get_weather],
    system_prompt="""
    你是天气助手。

工作流程：
1. 理解用户的城市查询
2. 使用 get_weather 工具获取数据
3. 简介清晰地回答问题

输出格式：
- 天气状况
- 温度
- 注意事项（如有）
    """
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "武汉的天气如何？"}
    ]
})
rprint(response)


{
    'messages': [
        HumanMessage(
            content='武汉的天气如何？',
            additional_kwargs={},
            response_metadata={},
            id='47b55bf8-8b36-4602-9f29-7483c417f675'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 196,
                    'total_tokens': 283,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 6,
                        'engine_ttft_ms': 29,
                        'engine_ttlt_ms': 696,
                        'pre_inference_ms': 177,
                        'service_tbt_ms': 6,
                        'service_ttft_ms': 611,
                        'service_ttlt_ms': 1268,
                        'user_visible_ttft_ms': 435
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EU9I7fGdPK1OAHJ2iq2GuxfToHiEj',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='Agent01',
            id='lc_run--01a0f73b-6f8d-73f1-8175-05f7462570cc-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '武汉'},
                    'id': 'call_w9Bonf2fhUedlRRWP0rcEXBt',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 196,
                'output_tokens': 87,
                'total_tokens': 283,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='武汉天气晴朗，温度：15℃',
            name='get_weather',
            id='b3f6dd7e-e281-4450-aaa8-a0863d093d27',
            tool_call_id='call_w9Bonf2fhUedlRRWP0rcEXBt'
        ),
        AIMessage(
            content='- 天气状况：晴朗，天空清澈，日照充足。  \n- 温度：约 15℃。  \n- 
注意事项（如有）：早晚偏凉，建议带薄外套；白天阳光较强，注意防晒和补水；如需长时间户外活动，建议查看即时空气质量与
后续逐小时预报。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 348,
                    'prompt_tokens': 237,
                    'total_tokens': 585,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 8,
                        'engine_ttft_ms': 28,
                        'engine_ttlt_ms': 3501,
                        'pre_inference_ms': 108,
                        'service_tbt_ms': 9,
                        'service_ttft_ms': 327,
                        'service_ttlt_ms': 3796,
                        'user_visible_ttft_ms': 219
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EU9I9FkFeOdV5d3o6ZMKeckg3y

## 3. 结构化输出

### 3.1. `Pydantic`

In [6]:
from pydantic import BaseModel, Field
import os
from rich import print as rprint
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_classic.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model = "gpt-5-mini",
    api_key = CLOSEAI_API_KEY,
    base_url = CLOSEAI_BASE_URL
)

class ContractInfo(BaseModel):
    """用户联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱")
    phone: str = Field(description="用户电话")

agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "从以下信息中提取用户信息：小明的邮箱是：xiaoming@163.com，电话是：13012344321，"}
    ]
})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息：小明的邮箱是：xiaoming@163.com，电话是：13012344321，',
            additional_kwargs={},
            response_metadata={},
            id='c2713ee5-c4b2-402b-a08a-03f48c2d7cf9'
        ),
        AIMessage(
            content='{"name":"小明","email":"xiaoming@163.com","phone":"13012344321"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 162,
                    'prompt_tokens': 234,
                    'total_tokens': 396,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 7,
                        'engine_ttft_ms': 31,
                        'engine_ttlt_ms': 1376,
                        'pre_inference_ms': 126,
                        'service_tbt_ms': 7,
                        'service_ttft_ms': 651,
                        'service_ttlt_ms': 1990,
                        'user_visible_ttft_ms': 525
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EU9gey7sIQklSeW2d50qhXsSYjtYX',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0f752-a50f-7651-adff-e573094bab46-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 234,
                'output_tokens': 162,
                'total_tokens': 396,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 128}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='xiaoming@163.com', phone='13012344321')
}

### 3.2. `ToolStrategy`

对于不知道原生结构化输出的模型，LangChain采用`ToolStrategy`工具调用的方式实现结构化的输出。

此策略兼容绝大多是支持工具调用的现代模型，其核心原理是动态创建一个“虚拟工具”，该工具的输入参数对应着期望的数据结构。

当模型需要生成最终答案时，系统会引导模型“调用”这个虚拟工具，从而简介产生符合要求的结构化数据。

In [7]:
from pydantic import BaseModel, Field
import os
from rich import print as rprint
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain_classic.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model = "gpt-5-mini",
    api_key = CLOSEAI_API_KEY,
    base_url = CLOSEAI_BASE_URL
)

class ContractInfo(BaseModel):
    """用户联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱")
    phone: str = Field(description="用户电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "从以下信息中提取用户信息：小明的邮箱是：xiaoming@163.com，电话是：13012344321，"}
    ]
})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息：小明的邮箱是：xiaoming@163.com，电话是：13012344321，',
            additional_kwargs={},
            response_metadata={},
            id='864b01e6-8b93-4eff-80db-f5d28f6f239e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 295,
                    'prompt_tokens': 168,
                    'total_tokens': 463,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 7,
                        'engine_ttft_ms': 42,
                        'engine_ttlt_ms': 2145,
                        'pre_inference_ms': 216,
                        'service_tbt_ms': 7,
                        'service_ttft_ms': 708,
                        'service_ttlt_ms': 2782,
                        'user_visible_ttft_ms': 492
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EU9otx1NA6eQ14skwgf19PqbrDe21',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0f75a-6fac-7453-9169-9c873ba8de5e-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'xiaoming@163.com', 'phone': '13012344321'},
                    'id': 'call_4tcX1KoO37zmUW1drxyLkKEJ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 168,
                'output_tokens': 295,
                'total_tokens': 463,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='xiaoming@163.com' phone='13012344321'",
            name='ContractInfo',
            id='7af35c80-056f-4c73-8a23-6f112ec7a5bc',
            tool_call_id='call_4tcX1KoO37zmUW1drxyLkKEJ'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='xiaoming@163.com', phone='13012344321')
}